# Ride Supply-Demand Gap Analysis

**Question:** Why do so many ride requests between the city and the airport go unfulfilled, where exactly is the gap,
and what should operations change first?

**Data:** 6,745 masked ride requests (11-15 July 2016, Mon-Fri), one row per request: pickup point (City / Airport),
driver ID, status (Trip Completed / Cancelled / No Cars Available), request and drop timestamps.

**Approach:** clean in Python → analyse in SQL (SQLite, `sql/analysis.sql`) → visualise → size the opportunity →
design an experiment to test the fix.

In [1]:
import sys, sqlite3, pandas as pd
sys.path.insert(0, "../src")
from prep import load, to_sqlite, load_queries, SLOT_ORDER
import charts
from IPython.display import display
pd.set_option("display.width", 200)

d = load("../data/uber_request_data.csv")
print(d.shape)
print("Missing values:\n", d.isna().sum())
d.head()

(6745, 12)
Missing values:
 request_id         0
pickup_point       0
driver_id       2650
status             0
request_ts         0
drop_ts         3914
hour               0
date               0
weekday            0
slot               0
completed          0
trip_min        3914
dtype: int64


,request_id,pickup_point,driver_id,status,request_ts,drop_ts,hour,date,weekday,slot,completed,trip_min
0,619,Airport,1.0,Trip Completed,2016-07-11 11:51:00,2016-07-11 13:00:00,11,2016-07-11,Monday,Daytime (10-16),1,69.000000
1,867,Airport,1.0,Trip Completed,2016-07-11 17:57:00,2016-07-11 18:47:00,17,2016-07-11,Monday,Evening Rush (17-21),1,50.000000
2,1807,City,1.0,Trip Completed,2016-07-12 09:17:00,2016-07-12 09:58:00,9,2016-07-12,Tuesday,Morning Rush (05-09),1,41.000000
3,2532,Airport,1.0,Trip Completed,2016-07-12 21:08:00,2016-07-12 22:03:00,21,2016-07-12,Tuesday,Evening Rush (17-21),1,55.000000
4,3112,City,1.0,Trip Completed,2016-07-13 08:33:16,2016-07-13 09:25:47,8,2016-07-13,Wednesday,Morning Rush (05-09),1,52.516667


### Data cleaning notes
* Timestamps came in two formats (`11/7/2016 11:51` and `13-07-2016 08:33:16`); both are parsed day-first.
* `driver_id` and `drop_ts` are empty by design for *No Cars Available* (no driver was assigned), and `drop_ts` is empty
  for *Cancelled* trips. These are not errors, so no rows are dropped.
* Each request is tagged with an hour and a time slot: Early Morning (00-04), Morning Rush (05-09), Daytime (10-16),
  Evening Rush (17-21), Late Night (22-23).

In [2]:
con = sqlite3.connect(":memory:")
to_sqlite(d, con)
Q = load_queries("../sql/analysis.sql")
R = {name: pd.read_sql(q, con) for name, q in Q.items()}
R["kpis"]

,requests,completed,fulfilment_pct,cancelled,no_cars,drivers,days
0,6745,2831,42.0,1264,2650,300,5


## 1. Where is the gap?

In [3]:
print(Q["fulfilment_by_pickup_slot"])
R["fulfilment_by_pickup_slot"]

SELECT pickup_point, slot,
       COUNT(*)                                            AS requests,
       SUM(completed)                                      AS completed,
       SUM(status = 'Cancelled')                           AS cancelled_n,
       SUM(status = 'No Cars Available')                   AS no_cars_n,
       ROUND(100.0 * AVG(completed), 1)                    AS fulfilment_pct,
       ROUND(100.0 * AVG(status = 'Cancelled'), 1)         AS cancelled_pct,
       ROUND(100.0 * AVG(status = 'No Cars Available'), 1) AS no_cars_pct
FROM requests
GROUP BY pickup_point, slot_order, slot
ORDER BY pickup_point, slot_order;


,pickup_point,slot,requests,completed,cancelled_n,no_cars_n,fulfilment_pct,cancelled_pct,no_cars_pct
0,Airport,Early Morning (00-04),253,103,2,148,40.7,0.8,58.5
1,Airport,Morning Rush (05-09),426,382,23,21,89.7,5.4,4.9
2,Airport,Daytime (10-16),478,327,64,87,68.4,13.4,18.2
3,Airport,Evening Rush (17-21),1800,373,106,1321,20.7,5.9,73.4
4,Airport,Late Night (22-23),281,142,3,136,50.5,1.1,48.4
5,City,Early Morning (00-04),325,111,63,151,34.2,19.4,46.5
6,City,Morning Rush (05-09),1677,472,820,385,28.1,48.9,23.0
7,City,Daytime (10-16),746,395,104,247,52.9,13.9,33.1
8,City,Evening Rush (17-21),542,411,60,71,75.8,11.1,13.1
9,City,Late Night (22-23),217,115,19,83,53.0,8.8,38.2


In [4]:
charts.fulfilment_heatmap(R["fulfilment_by_pickup_slot"], SLOT_ORDER, "../images/3_fulfilment_heatmap.png");

In [5]:
charts.status_mix(R["fulfilment_by_pickup_slot"], SLOT_ORDER, "../images/2_status_mix.png");

In [6]:
print(Q["lost_requests_ranked"])
R["lost_requests_ranked"]

-- Where are unfulfilled requests concentrated? (window functions)
WITH lost AS (
    SELECT pickup_point, slot, slot_order,
           COUNT(*)                          AS lost,
           SUM(status = 'Cancelled')         AS cancelled,
           SUM(status = 'No Cars Available') AS no_cars
    FROM requests
    WHERE completed = 0
    GROUP BY pickup_point, slot, slot_order
)
SELECT pickup_point, slot, lost, cancelled, no_cars,
       ROUND(100.0 * lost / SUM(lost) OVER (), 1)                         AS pct_of_all_lost,
       ROUND(100.0 * SUM(lost) OVER (ORDER BY lost DESC) / SUM(lost) OVER (), 1) AS cumulative_pct,
       RANK() OVER (ORDER BY lost DESC)                                   AS rank
FROM lost
ORDER BY rank;


,pickup_point,slot,lost,cancelled,no_cars,pct_of_all_lost,cumulative_pct,rank
0,Airport,Evening Rush (17-21),1427,106,1321,36.5,36.5,1
1,City,Morning Rush (05-09),1205,820,385,30.8,67.2,2
2,City,Daytime (10-16),351,104,247,9.0,76.2,3
3,City,Early Morning (00-04),214,63,151,5.5,81.7,4
4,Airport,Daytime (10-16),151,64,87,3.9,85.5,5
5,Airport,Early Morning (00-04),150,2,148,3.8,89.4,6
6,Airport,Late Night (22-23),139,3,136,3.6,92.9,7
7,City,Evening Rush (17-21),131,60,71,3.3,96.3,8
8,City,Late Night (22-23),102,19,83,2.6,98.9,9
9,Airport,Morning Rush (05-09),44,23,21,1.1,100.0,10


**Finding 1. Two time slots cause two-thirds of all failed requests.**
* **Airport pickups, Evening Rush (17-21):** 1,800 requests, only **20.7% fulfilled**; 73% fail with *No Cars Available*.
* **City pickups, Morning Rush (05-09):** 1,677 requests, only **28.1% fulfilled**; **49% are cancelled by drivers**.
* Together they account for **67.2% of all 3,914 unfulfilled requests**. Overall fulfilment is just **42.0%**.
* The two failures are different problems: in the morning **cars exist but drivers refuse the trip**; in the evening
  **there are simply no cars at the airport**.

## 2. Why? Demand runs in one direction at a time

In [7]:
charts.hourly_demand_supply(R["hourly_demand_supply"], "../images/1_hourly_demand_supply.png");

In [8]:
print(Q["directional_imbalance"])
R["directional_imbalance"]

-- Requests leaving the city vs leaving the airport in each time slot
SELECT slot,
       SUM(pickup_point = 'City')    AS city_to_airport,
       SUM(pickup_point = 'Airport') AS airport_to_city,
       ROUND(1.0 * SUM(pickup_point = 'City') / SUM(pickup_point = 'Airport'), 2) AS city_airport_ratio
FROM requests
GROUP BY slot_order, slot
ORDER BY slot_order;


,slot,city_to_airport,airport_to_city,city_airport_ratio
0,Early Morning (00-04),325,253,1.28
1,Morning Rush (05-09),1677,426,3.94
2,Daytime (10-16),746,478,1.56
3,Evening Rush (17-21),542,1800,0.30
4,Late Night (22-23),217,281,0.77


In [9]:
charts.imbalance(R["directional_imbalance"], "../images/4_directional_imbalance.png");

**Finding 2. The gap is a direction problem, not a total-supply problem.**
* In the Morning Rush there are **3.94 city-to-airport requests for every airport-to-city request**. A driver who takes
  a city passenger to the airport at 7 am finds few passengers going back, so the trip often means a long unpaid wait or
  an empty return. **Likely reason drivers cancel city pickups in the morning.**
* In the Evening Rush it flips to **0.30** (about 3.3 airport requests for each city request). Few cars went to the airport
  in the afternoon, so there are **no cars waiting when evening flights land**.
* Trip duration is the same from both sides (median ~52 min), so trip length itself does not explain the gap.

*Caveat:* the data has no driver location or flight schedule, so the "long wait at the airport" explanation is a
hypothesis consistent with the data, which the experiment in section 5 is designed to test.

In [10]:
d.groupby("pickup_point").trip_min.describe().round(1)

,count,mean,std,min,25%,50%,75%,max
pickup_point,,,,,,,,
Airport,1327.0,52.2,13.8,21.2,41.0,51.6,64.0,82.1
City,1504.0,52.6,13.9,20.8,41.0,53.0,63.9,83.0


In [11]:
R["daily_trend"]

,date,weekday,requests,fulfilment_pct
0,2016-07-11,Monday,1367,44.0
1,2016-07-12,Tuesday,1307,43.0
2,2016-07-13,Wednesday,1337,43.2
3,2016-07-14,Thursday,1353,39.2
4,2016-07-15,Friday,1381,40.6


The pattern holds on every weekday (fulfilment 39-44% each day), so it is structural, not a one-off bad day.

## 3. Driver (partner) view

In [12]:
print(Q["driver_quintiles"])
R["driver_quintiles"]

-- Are cancellations spread evenly, or driven by a small group of drivers?
WITH d AS (
    SELECT driver_id, COUNT(*) AS assigned, SUM(completed) AS trips,
           SUM(status = 'Cancelled') AS cancels
    FROM requests WHERE driver_id IS NOT NULL GROUP BY driver_id
), q AS (
    SELECT *, NTILE(5) OVER (ORDER BY 1.0 * cancels / assigned DESC) AS quintile FROM d
)
SELECT quintile,
       COUNT(*)                                                     AS drivers,
       ROUND(100.0 * SUM(cancels) / SUM(assigned), 1)               AS cancel_rate_pct,
       SUM(cancels)                                                 AS cancels,
       ROUND(100.0 * SUM(cancels) / (SELECT SUM(cancels) FROM d), 1) AS pct_of_all_cancels,
       ROUND(AVG(trips), 1)                                         AS avg_trips_per_driver
FROM q
GROUP BY quintile
ORDER BY quintile;


,quintile,drivers,cancel_rate_pct,cancels,pct_of_all_cancels,avg_trips_per_driver
0,1,60,48.4,398,31.5,7.1
1,2,60,36.7,313,24.8,9.0
2,3,60,29.5,243,19.2,9.7
3,4,60,23.7,203,16.1,10.9
4,5,60,14.4,107,8.5,10.6


In [13]:
charts.driver_quintiles(R["driver_quintiles"], "../images/5_driver_quintiles.png");

In [14]:
sc = R["driver_scorecard"]
print("Drivers:", len(sc), "| avg trips per driver over 5 days:", round(sc.trips.mean(), 1))
print("Drivers cancelling more than 40% of assigned trips:", (sc.cancel_rate_pct > 40).sum())
sc.head(10)

Drivers: 300 | avg trips per driver over 5 days: 9.4
Drivers cancelling more than 40% of assigned trips: 63


,driver_id,assigned,trips,cancels,cancel_rate_pct,cancel_quintile
0,267.0,15,6,9,60.0,1
1,206.0,17,7,10,58.8,1
2,103.0,12,5,7,58.3,1
3,104.0,12,5,7,58.3,1
4,54.0,19,8,11,57.9,1
5,84.0,21,9,12,57.1,1
6,131.0,14,6,8,57.1,1
7,135.0,11,5,6,54.5,1
8,62.0,13,6,7,53.8,1
9,229.0,13,6,7,53.8,1


**Finding 3. Cancellations are concentrated in a group of drivers.**
* The fifth of drivers who cancel most (60 drivers) cancel **48% of their assigned trips** and cause **31.5% of all
  cancellations**; the best fifth cancel only 14%.
* **63 of 300 drivers** cancel more than 40% of trips assigned to them, a clear list for targeted nudges or coaching.
* Drivers average only ~9.4 completed trips in 5 days, so there is spare driver capacity to re-position.

## 4. Sizing the opportunity

In [15]:
t = R["fulfilment_by_pickup_slot"].set_index(["pickup_point", "slot"])
city_am = t.loc[("City", "Morning Rush (05-09)")]
air_pm = t.loc[("Airport", "Evening Rush (17-21)")]
days = R["kpis"].days[0]
base = R["kpis"].completed[0]; total = R["kpis"].requests[0]

a = city_am.cancelled_n * 0.5                        # A: halve morning city cancellations
b = air_pm.requests * 0.50 - air_pm.completed        # B: airport evening fulfilment 20.7% -> 50% (late-night level)
scen = pd.DataFrame({
    "lever": ["A. Halve City Morning-Rush cancellations", "B. Lift Airport Evening-Rush fulfilment to 50%", "A + B"],
    "extra_trips_5_days": [a, b, a + b],
})
scen["extra_trips_per_day"] = (scen.extra_trips_5_days / days).round(0)
scen["overall_fulfilment_pct"] = ((base + scen.extra_trips_5_days) / total * 100).round(1)
print("Baseline overall fulfilment:", round(base / total * 100, 1), "%")
scen

Baseline overall fulfilment: 42.0 %


,lever,extra_trips_5_days,extra_trips_per_day,overall_fulfilment_pct
0,A. Halve City Morning-Rush cancellations,410.0,82.0,48.1
1,B. Lift Airport Evening-Rush fulfilment to 50%,527.0,105.0,49.8
2,A + B,937.0,187.0,55.9


Assumes each recovered request becomes a completed trip. The 50% target for airport evenings is the level the airport
already reaches late at night, so it is a realistic first goal rather than a best case. Both levers together would lift
overall fulfilment from **42.0% to about 55.9%**, roughly **187 more completed trips per day**.

## 5. Recommendations and the experiment to test them

| # | Problem | Action | Metric to move |
|---|---|---|---|
| 1 | Airport has no cars 17-21 | Pull supply to the airport before 17:00: an incentive for drivers who are at the airport during the evening peak, and dynamic pricing at the airport 17-21 | Airport evening fulfilment %, No-Cars % |
| 2 | Drivers cancel city-to-airport trips 05-09 | Make the morning airport trip worth it: return-trip priority / guaranteed queue spot at the airport, or a bonus per morning airport drop | City morning cancellation % |
| 3 | 20% of drivers cause 31.5% of cancellations | Weekly cancellation scorecard, nudges and coaching for the 63 drivers above 40%; track if they improve | Driver cancellation rate |
| 4 | Ongoing | Daily dashboard of fulfilment by pickup point x time slot, with alerts when a slot drops below target | Fulfilment % by slot |

### Experiment design: airport evening supply incentive
* **Hypothesis:** paying drivers to be at the airport for the evening peak lifts airport evening fulfilment from ~21%.
* **Design:** *switchback* by day (alternate incentive days and control days). Drivers share one supply pool, so a
  driver-level A/B split would leak; switching whole days avoids that.
* **Primary metric:** airport fulfilment % 17-21. **Guardrails:** city evening fulfilment, cost per extra trip, driver
  earnings per hour.
* **Sample size** (below): requests needed per arm to detect a lift from 21% to 30%.

In [16]:
from math import sqrt, ceil
p1, p2, za, zb = 0.207, 0.30, 1.96, 0.84          # alpha 0.05 two-sided, power 80%
pbar = (p1 + p2) / 2
n = ((za * sqrt(2 * pbar * (1 - pbar)) + zb * sqrt(p1 * (1 - p1) + p2 * (1 - p2))) / (p2 - p1)) ** 2
per_evening = air_pm.requests / days
print(f"Requests needed per arm: {ceil(n)}  |  airport evening requests per day: {per_evening:.0f}")
print(f"=> about {ceil(n / per_evening)} evenings per arm in theory; run 7 incentive + 7 control evenings (2 weeks)")
print("   to cover day-to-day variation, since days (not requests) are the unit that is switched.")

Requests needed per arm: 342  |  airport evening requests per day: 360
=> about 1 evenings per arm in theory; run 7 incentive + 7 control evenings (2 weeks)
   to cover day-to-day variation, since days (not requests) are the unit that is switched.


**Decision rule:** roll out if airport evening fulfilment rises by at least 8 points with cost per extra trip below
the average fare margin, and city evening fulfilment does not fall.